In [1]:
!pip install dagshub mlflow xgboost scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
from mlflow.tracking import MlflowClient
warnings.filterwarnings('ignore')

dagshub.init(repo_owner='sophyrise', repo_name='ieee-cis-fraud-detection', mlflow=True)
client = MlflowClient()
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=28f98a5b-3a07-41ee-8abf-36fa2c8402ae&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c5beb5e8cede555d421279c3d2eee5afb957140f5200adfa21c5c8fd4f0eedc1




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


In [3]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

test_txn = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_id  = pd.read_csv(f"{DATA_DIR}/test_identity.csv")
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

test = test_txn.merge(test_id, on="TransactionID", how="left")
transaction_ids = test["TransactionID"].copy()
X_test_raw = test.drop(columns=["TransactionID"])

del test, test_txn, test_id
gc.collect()

print(f"Test shape: {X_test_raw.shape}")
print(f"Transaction IDs: {len(transaction_ids)}")

Test shape: (506691, 432)
Transaction IDs: 506691


In [4]:
experiments = {
    "XGBoost":                     ("XGB_Final_Pipeline",  "XGBoost_FraudDetection"),
    "Random Forest":               ("RF_Final_Pipeline",   "RandomForest_FraudDetection"),
    "Decision Trees":              ("DT_Final_Pipeline",   "DecisionTree_FraudDetection"),
    "LogisticRegression-Training": ("LR_Final_Pipeline",   "LogisticRegression_FraudDetection"),
    "Neural Networks":             ("MLP_Final_Pipeline",  "MLP_FraudDetection"),
    "LightGBM":                    ("LGB_Final_Pipeline",  "LightGBM_FraudDetection"),
    "Gradient Boosting":           ("GB_Final_Pipeline",   "GradientBoosting_FraudDetection"),
}

results = []
for exp_name, (run_name, registry_name) in experiments.items():
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"  ⚠️  not found: {exp_name}")
        continue

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    if not runs:
        print(f"  ⚠️  run not found: {run_name} in {exp_name}")
        continue

    run  = runs[0]
    auc  = run.data.metrics.get("val_auc") or run.data.metrics.get("cv_mean_auc")
    metric_used = "val_auc" if "val_auc" in run.data.metrics else "cv_mean_auc"

    results.append({
        "model":         exp_name,
        "run_id":        run.info.run_id,
        "auc":           auc,
        "metric_used":   metric_used,
        "registry_name": registry_name,
    })
    print(f"  {exp_name:<35} | AUC: {auc:.4f} ({metric_used})")

results_df = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
print("\n📊 Model Ranking:")
print(results_df[["model", "auc", "metric_used"]].to_string())

best = results_df.iloc[0]
print(f"\n✅ Best model: {best['model']}")
print(f"   AUC: {best['auc']:.4f} | Run ID: {best['run_id']}")

  XGBoost                             | AUC: 0.9672 (val_auc)
  Random Forest                       | AUC: 0.9475 (val_auc)
  Decision Trees                      | AUC: 0.8667 (val_auc)
  LogisticRegression-Training         | AUC: 0.8807 (val_auc)
  Neural Networks                     | AUC: 0.8787 (val_auc)
  LightGBM                            | AUC: 0.9639 (val_auc)
  Gradient Boosting                   | AUC: 0.9200 (val_auc)

📊 Model Ranking:
                         model      auc metric_used
0                      XGBoost  0.96724     val_auc
1                     LightGBM  0.96392     val_auc
2                Random Forest  0.94747     val_auc
3            Gradient Boosting  0.91999     val_auc
4  LogisticRegression-Training  0.88069     val_auc
5              Neural Networks  0.87874     val_auc
6               Decision Trees  0.86667     val_auc

✅ Best model: XGBoost
   AUC: 0.9672 | Run ID: cd0a105927694771b66453f103b591ab


In [5]:
model = mlflow.sklearn.load_model(f"models:/{best['registry_name']}/latest")
print(f"✅ Model loaded: {type(model.named_steps['clf']).__name__}")
print(f"   Loaded from registry: {best['registry_name']}")

✅ Model loaded: XGBClassifier
   Loaded from registry: XGBoost_FraudDetection


In [6]:
train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
train_id.columns = [c.replace('-', '_') for c in train_id.columns]
train = train_txn.merge(train_id, on="TransactionID", how="left")
X_train_raw = train.drop(columns=["isFraud", "TransactionID"])
del train, train_txn, train_id
gc.collect()

TXN_MISSING_THRESHOLD   = 0.80
ID_MISSING_THRESHOLD    = 0.95
NEAR_CONSTANT_THRESHOLD = 0.99
CORR_THRESHOLD          = 0.98

def preprocess(X_train, X_test):
    X_train = X_train.copy()
    X_test  = X_test.copy()

    id_cols  = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_cols = [c for c in X_train.columns if c not in id_cols]

    missing = X_train.isnull().mean()
    drop_missing = (
        [c for c in txn_cols if missing[c] > TXN_MISSING_THRESHOLD] +
        [c for c in id_cols  if missing[c] > ID_MISSING_THRESHOLD]
    )
    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    near_const = [
        c for c in X_train.columns
        if X_train[c].value_counts(dropna=False, normalize=True).iloc[0] > NEAR_CONSTANT_THRESHOLD
    ]
    X_train = X_train.drop(columns=near_const)
    X_test  = X_test.drop(columns=[c for c in near_const if c in X_test.columns])

    for df in [X_train, X_test]:
        df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"].clip(lower=0))
        df["hour_sin"] = np.sin(2 * np.pi * ((df["TransactionDT"] // 3600) % 24) / 24)
        df["hour_cos"] = np.cos(2 * np.pi * ((df["TransactionDT"] // 3600) % 24) / 24)
    X_train = X_train.drop(columns=["TransactionDT"])
    X_test  = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols)
    X_test  = X_test.drop(columns=[c for c in const_cols if c in X_test.columns])

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    idx  = np.random.RandomState(42).choice(len(X_train), size=min(120_000, len(X_train)), replace=False)
    corr  = X_train.iloc[idx][num_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]
    X_train = X_train.drop(columns=drop_corr)
    X_test  = X_test.drop(columns=[c for c in drop_corr if c in X_test.columns])

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_test

X_test_processed = preprocess(X_train_raw, X_test_raw)
del X_train_raw
gc.collect()

print(f"✅ Test processed: {X_test_processed.shape}")
print(f"   NaNs: {X_test_processed.isnull().sum().sum()}")

✅ Test processed: (506691, 301)
   NaNs: 0


In [7]:
predictions = model.predict_proba(X_test_processed)[:, 1]

print(f"Predictions shape: {predictions.shape}")
print(f"Prediction range:  [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Mean fraud probability: {predictions.mean():.4f}")

Predictions shape: (506691,)
Prediction range:  [0.0000, 0.9999]
Mean fraud probability: 0.0891


In [8]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud":       predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"✅ Submission saved: {submission.shape}")
print(submission.head(10).to_string())

✅ Submission saved: (506691, 2)
   TransactionID   isFraud
0        3663549  0.013351
1        3663550  0.012298
2        3663551  0.001647
3        3663552  0.004343
4        3663553  0.006580
5        3663554  0.019545
6        3663555  0.027744
7        3663556  0.173051
8        3663557  0.000222
9        3663558  0.051207


In [9]:
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

assert len(submission) == len(sample_sub),                   "❌ Row count mismatch!"
assert list(submission.columns) == list(sample_sub.columns), "❌ Column mismatch!"
assert submission["isFraud"].between(0, 1).all(),            "❌ Predictions out of [0,1] range!"

print("=== Inference Summary ===")
print(f"Best model    : {best['model']}")
print(f"Registry name : {best['registry_name']}/latest")
print(f"Val AUC       : {best['auc']:.4f}")
print(f"Model type    : {type(model.named_steps['clf']).__name__}")
print(f"Features used : {X_test_processed.shape[1]}")
print(f"Predictions   : {len(submission)} transactions")
print(f"Fraud range   : {predictions.min():.4f} — {predictions.max():.4f}")
print(f"Mean fraud    : {predictions.mean():.4f}")
print("✅ Submission valid — ready to upload to Kaggle!")

=== Inference Summary ===
Best model    : XGBoost
Registry name : XGBoost_FraudDetection/latest
Val AUC       : 0.9672
Model type    : XGBClassifier
Features used : 301
Predictions   : 506691 transactions
Fraud range   : 0.0000 — 0.9999
Mean fraud    : 0.0891
✅ Submission valid — ready to upload to Kaggle!
